# Paper 3 — M3: FR2 Layer, Angular Reports, Cluster-Identification Preview

**Milestone M3** (spec v0.2): the adopted A12 geometry at **28 GHz / 400 MHz**, on the SAME outdoor
UE positions as the M1 FR1 dataset — so the FR1/FR2 pairing for E3 comes for free from the existing
M1 HDF5, and only the FR2 layer is traced here.

| | FR1 (from M1) | FR2 (this notebook) |
|---|---|---|
| Carrier / BW | 3.5 GHz / 50 MHz | 28 GHz / 400 MHz |
| Array | 4x2 | 8x8 panel |
| UE | 1x2 | 2x2 panel (UE-frame AoA) |
| Delay resolution | 20 ns | 2.5 ns |

**Pre-registered M3 gate** (fixed before tracing): (i) FR2 outage — UEs hearing no array — below
20% of the outdoor set; (ii) median FR2 cluster size <= median FR1 cluster size at the same
gamma = -10 dB (the confinement premise of H4: at FR2, coverage itself localizes); (iii) per-batch
runtime within T4 budget.

Materials note: Sionna's ITU radio materials implement the ITU-R P.2040 frequency-dependent model,
valid through mmWave; the same scene is therefore reusable at 28 GHz without material edits.

UE-frame AoA note: the azimuth sector is estimated from the phase across the UE's horizontal panel
pair, i.e. in the UE's own reference frame — which is what a real terminal can measure without an
absolute orientation. The simulation keeps a fixed panel orientation; this idealization is stated
in the paper.

In [ ]:
# %% [1] Environment setup — safe to re-run; auto-restarts ONCE after installing
import importlib.util, subprocess, sys

if importlib.util.find_spec('sionna') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sionna==2.0.1', 'h5py'], check=True)
    print('Install done — the runtime will now restart (EXPECTED).')
    print('When it comes back, run this cell again: it will skip the install.')
    import os, time; time.sleep(1); os.kill(os.getpid(), 9)

!nvidia-smi -L
import sionna, sionna.rt
print('Sionna:', sionna.__version__, '| Sionna RT:', sionna.rt.__version__)

In [ ]:
# %% [2] Drive mount + canary
import os, uuid, glob, json
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/paper3'
M1_DIR, M2_DIR = os.path.join(BASE, 'm1'), os.path.join(BASE, 'm2')
M3_DIR = os.path.join(BASE, 'm3'); CKPT_DIR = os.path.join(M3_DIR, 'ckpt')
os.makedirs(CKPT_DIR, exist_ok=True)

def drive_canary(path=M3_DIR):
    token = str(uuid.uuid4()); c = os.path.join(path, '.canary')
    with open(c, 'w') as f: f.write(token)
    with open(c) as f: assert f.read() == token, 'DRIVE CANARY FAILED: phantom mount.'
    os.remove(c); print('Drive canary OK:', path)

drive_canary()

In [ ]:
# %% [3] Configuration
from dataclasses import dataclass
import numpy as np

@dataclass
class M3Config:
    # --- FR2 band ---
    carrier_hz: float = 28e9
    bandwidth_hz: float = 400e6
    num_subcarriers: int = 256
    aoa_subcarriers: int = 16              # coarse grid used only for the UE-frame AoA estimate
    # --- geometry: identical to M1/M2 (A12) ---
    area_center: tuple = (0.0, 50.0)
    area_half: float = 100.0
    ue_height: float = 1.5
    tx_height: float = 7.0
    n_outer: int = 8
    n_inner: int = 4
    tx_ring_radius: float = 80.0
    inner_ring_radius: float = 35.0
    tx_ring_jitter: float = 12.0
    # --- sampling: first N of the SAME outdoor UE set as M1 (loaded from Drive) ---
    n_points: int = 3000
    batch_size: int = 24                   # FR2 tensors are larger: smaller batches
    # --- ray tracing / reports ---
    max_depth: int = 5
    gamma_db: float = -10.0
    rsrp_quant_db: float = 1.0
    n_sectors: int = 4                     # UE-frame azimuth quantization
    max_neighbors: int = 8
    seed: int = 2026

CFG = M3Config()
SC_SPACING = CFG.bandwidth_hz / CFG.num_subcarriers
TOA_QUANT = 1.0 / CFG.bandwidth_hz
RUN_TAG = f'M3_fr2_A12_N{CFG.n_points}_seed{CFG.seed}'
print(f'{RUN_TAG} | subcarrier spacing {SC_SPACING/1e6:.2f} MHz | delay resolution {TOA_QUANT*1e9:.1f} ns')

In [ ]:
# %% [4] FR2 scene on the A12 geometry (TX positions from M2's tagged file, else regenerated)
import mitsuba as mi
from sionna.rt import (load_scene, PlanarArray, Transmitter, Receiver,
                       PathSolver, subcarrier_frequencies)

scene = load_scene(sionna.rt.scene.munich, merge_shapes=True)
scene.frequency = CFG.carrier_hz
scene.tx_array = PlanarArray(num_rows=8, num_cols=8, vertical_spacing=0.5,
                             horizontal_spacing=0.5, pattern='tr38901', polarization='V')
scene.rx_array = PlanarArray(num_rows=2, num_cols=2, vertical_spacing=0.5,
                             horizontal_spacing=0.5, pattern='iso', polarization='V')

tx_file = os.path.join(M2_DIR, 'tx_positions_A12.npy')
if os.path.exists(tx_file):
    TX_POS = np.load(tx_file); print('TX positions loaded from', tx_file)
else:
    def ring(n, radius, offset, cfg, rng):
        ang = np.linspace(0, 2*np.pi, n + 1)[:-1] + offset + rng.uniform(-0.15, 0.15, n)
        r = radius + rng.uniform(-cfg.tx_ring_jitter, cfg.tx_ring_jitter, n)
        return np.stack([cfg.area_center[0] + r*np.cos(ang),
                         cfg.area_center[1] + r*np.sin(ang), np.full(n, cfg.tx_height)], 1)
    rng = np.random.default_rng(CFG.seed)
    TX_POS = np.concatenate([ring(CFG.n_outer, CFG.tx_ring_radius, 0.0, CFG, rng),
                             ring(CFG.n_inner, CFG.inner_ring_radius, np.pi/CFG.n_inner, CFG, rng)])
    TX_POS[7] = [29.7, 0.4, CFG.tx_height]     # tx_7 relocation found in M1 (deterministic search)
    print('TX positions regenerated from seed (M2 file not found)')

A = TX_POS.shape[0]
for a, p in enumerate(TX_POS):
    tx = Transmitter(name=f'tx_{a}', position=[float(v) for v in p])
    scene.add(tx)
    tx.look_at([CFG.area_center[0], CFG.area_center[1], CFG.ue_height])
FREQS = subcarrier_frequencies(CFG.num_subcarriers, SC_SPACING)
AOA_FREQS = subcarrier_frequencies(CFG.aoa_subcarriers, CFG.bandwidth_hz/CFG.aoa_subcarriers)
SOLVER = PathSolver()
print(f'{A} arrays at 28 GHz, 8x8 panels')

In [ ]:
# %% [5] UE positions: the SAME outdoor set as the M1 FR1 dataset (pairing is the point)
ue_file = os.path.join(M1_DIR, 'ue_positions_A12_N5000_outdoor_seed2026.npy')
assert os.path.exists(ue_file), f'{ue_file} missing — the M1 outdoor set is the pairing anchor'
UE_POS = np.load(ue_file)[:CFG.n_points]
print('UE positions:', UE_POS.shape, '(first', CFG.n_points, 'of the M1 outdoor set)')

In [ ]:
# %% [6] Batched FR2 tracing with checkpoints: energies, first-path ToA, delay spread, UE-frame AoA
import time
RUN_CKPT = os.path.join(CKPT_DIR, RUN_TAG); os.makedirs(RUN_CKPT, exist_ok=True)

def solve_batch(positions):
    names = []
    for i, p in enumerate(positions):
        scene.add(Receiver(name=f'rx_{i}', position=[float(v) for v in p])); names.append(f'rx_{i}')
    paths = SOLVER(scene, max_depth=CFG.max_depth)
    a_cir, tau = paths.cir(normalize_delays=False, out_type='numpy')
    a_cir, tau = np.asarray(a_cir), np.asarray(tau)
    # UE-frame AoA: phase across the UE's horizontal element pair, from a coarse-subcarrier CFR
    h = np.asarray(paths.cfr(frequencies=AOA_FREQS, normalize_delays=False, out_type='numpy'))
    h = h.squeeze(-2).sum(3)/np.sqrt(scene.tx_array.array_size)     # [B, rx_ant(4), A, S_aoa]
    for n in names: scene.remove(n)
    p_path = (np.abs(a_cir[..., 0])**2).sum(axis=(1, 3))            # [B, A, P]
    tau_l = tau if tau.ndim == 3 else tau[:, 0, :, 0, :]            # [B, A, P]
    valid = tau_l >= 0
    E = 10*np.log10(np.where(valid, p_path, 0).sum(-1) + 1e-30)     # [B, A]
    toa = np.where(valid, tau_l, np.inf).min(-1)                    # first path [B, A]
    w = np.where(valid, p_path, 0.0); ws = w.sum(-1) + 1e-30
    tm = (w*np.where(valid, tau_l, 0)).sum(-1)/ws
    ds = np.sqrt((w*(np.where(valid, tau_l, 0)-tm[..., None])**2).sum(-1)/ws)
    # horizontal pair of the 2x2 panel: elements 0 and 1 (same row); dphi -> azimuth sector
    x01 = (h[:, 0]*np.conj(h[:, 1])).sum(-1)                        # [B, A]
    dphi = np.angle(x01)
    sect = np.floor((dphi + np.pi)/(2*np.pi)*CFG.n_sectors).astype(np.int8)
    return E.astype(np.float32), toa.astype(np.float32), ds.astype(np.float32),            dphi.astype(np.float32), sect

n_b = int(np.ceil(CFG.n_points/CFG.batch_size))
done = {int(f.split('_')[-1].split('.')[0]) for f in glob.glob(os.path.join(RUN_CKPT, 'b_*.npz'))}
print(f'[{RUN_TAG}] {n_b} batches | {len(done)} done')
t0 = time.time()
for b in range(n_b):
    if b in done: continue
    sl = slice(b*CFG.batch_size, min((b+1)*CFG.batch_size, CFG.n_points))
    E, toa, ds, dphi, sect = solve_batch(UE_POS[sl])
    drive_canary(RUN_CKPT)
    np.savez_compressed(os.path.join(RUN_CKPT, f'b_{b:04d}.npz'),
                        E=E, toa=toa, ds=ds, dphi=dphi, sect=sect, idx=np.arange(sl.start, sl.stop))
    el = time.time()-t0
    print(f'batch {b+1}/{n_b} | {el:5.1f} s elapsed | ETA {el/(b+1-len(done))*(n_b-b-1):5.0f} s')

In [ ]:
# %% [7] Assemble FR2 + pair with the FR1 reports from the M1 HDF5 (position-matched)
import h5py
files = sorted(glob.glob(os.path.join(RUN_CKPT, 'b_*.npz')))
def cat(k): return np.concatenate([np.load(f)[k] for f in files])
IDX = cat('idx'); o = np.argsort(IDX)
E2, TOA2, DS2, DPHI2, SECT2 = (cat(k)[o] for k in ('E', 'toa', 'ds', 'dphi', 'sect'))
print('FR2 features:', E2.shape)

heard2 = np.isfinite(TOA2) & (E2 >= E2.max(1, keepdims=True) - 40)   # 40 dB receiver dynamic range
outage2 = ~heard2.any(1)
mem2 = (E2 >= E2.max(1, keepdims=True) + CFG.gamma_db) & heard2  # FR2 gamma-cluster
strongest2 = np.where(heard2, E2, -np.inf).argmax(1)
dtau2 = TOA2 - np.take_along_axis(TOA2, strongest2[:, None], 1)
dtau2_q = np.round(dtau2/TOA_QUANT)*TOA_QUANT
E2_q = np.round(E2/CFG.rsrp_quant_db)*CFG.rsrp_quant_db

m1_h5 = os.path.join(M1_DIR, 'paper3_static_fr1_A12_N5000_outdoor_seed2026.h5')
with h5py.File(m1_h5, 'r') as f:
    pos1 = f['positions'][:]
    E1_q = f['reports/E_dB_q'][:]; dtau1_q = f['reports/dtau_q'][:]
    ds1 = f['reports/sigma_tau'][:]; mem1 = f['cluster_gt/gamma'][:]
key = {tuple(np.round(p, 3)): i for i, p in enumerate(pos1)}
map1 = np.array([key.get(tuple(np.round(p, 3)), -1) for p in UE_POS])
ok = map1 >= 0
print(f'FR1 pairing: {ok.sum()}/{len(ok)} UEs matched to the M1 dataset '
      f'({(~ok).sum()} were dropped by M1 coverage)')

In [ ]:
# %% [8] M3 gate — pre-registered criteria
sz2 = mem2[ok].sum(1); sz1 = mem1[map1[ok]].sum(1)
checks = {
    'FR2 outage < 20%': outage2[ok].mean() < 0.20,
    'FR2 confinement: median cluster size <= FR1': np.median(sz2) <= np.median(sz1),
}
print(f'FR2 outage: {outage2[ok].mean()*100:.1f}% | cluster size median FR2 {np.median(sz2):.0f} '
      f'vs FR1 {np.median(sz1):.0f} | FR2 singletons {np.mean(sz2==1)*100:.1f}% '
      f'vs FR1 {np.mean(sz1==1)*100:.1f}%')
for k, v in checks.items(): print(f'[{"PASS" if v else "FAIL"}] {k}')
print('M3 GATE:', 'PASSED' if all(checks.values()) else 'NOT PASSED')

In [ ]:
# %% [9] E3 preview: kNN localization from report features, FR1 vs FR2 vs both
# Not the full E3 (that gets its own notebook): a first read of how informative the report
# vectors are, band by band, on the shared UE set. Positions are used only as regression target.
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split

def feat_fr1(i): return np.concatenate([E1_q[i], np.nan_to_num(dtau1_q[i], posinf=0)* 1e6, ds1[i]*1e9], 1)
def feat_fr2(m): return np.concatenate([E2_q[m], np.nan_to_num(dtau2_q[m], posinf=0)*1e6,
                                        DS2[m]*1e9, DPHI2[m]], 1)
i1 = map1[ok]
sets = {'FR1': feat_fr1(i1), 'FR2': feat_fr2(ok), 'FR1+FR2': np.concatenate([feat_fr1(i1), feat_fr2(ok)], 1)}
y = UE_POS[ok][:, :2]
for name, X in sets.items():
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=CFG.seed)
    err = np.linalg.norm(KNeighborsRegressor(5).fit(Xtr, ytr).predict(Xte) - yte, axis=1)
    print(f'{name:8s} | kNN localization error: median {np.median(err):5.2f} m | P90 {np.percentile(err, 90):5.2f} m')

In [ ]:
# %% [9b] Fixed evaluation: sane audibility, standardized kNN, FR2 TDoA multilateration
# Cell [9] had three instrumentation flaws, corrected here:
#  (1) 120 dB audibility counted through-wall refracted whispers -> 40 dB receiver dynamic range;
#  (2) unscaled kNN features let the dB energies dominate the microsecond ToAs, so every feature
#      set degenerated into RSS fingerprinting and the 2.5 ns FR2 resolution was invisible;
#  (3) kNN is floored by database density (~3.6 m spacing at N=3000): sub-meter accuracy must come
#      from GEOMETRY (TDoA multilateration over the cluster arrays), not from fingerprint recall.
C_LIGHT = 3e8
heard40 = np.isfinite(TOA2) & (E2 >= E2.max(1, keepdims=True) - 40)     # 40 dB dynamic range
outage40 = ~heard40.any(1)
mem40 = (E2 >= E2.max(1, keepdims=True) + CFG.gamma_db) & heard40
sz40 = mem40[ok].sum(1)
print(f'FR2 with 40 dB audibility: outage {outage40[ok].mean()*100:.1f}% | '
      f'cluster median {np.median(sz40):.0f} | singletons {np.mean(sz40==1)*100:.1f}% '
      f'(FR1: median {np.median(mem1[map1[ok]].sum(1)):.0f}, '
      f'singletons {np.mean(mem1[map1[ok]].sum(1)==1)*100:.1f}%)')

# --- standardized kNN (z-score per feature) ---
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
for name, X in sets.items():
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=CFG.seed)
    mdl = make_pipeline(StandardScaler(), KNeighborsRegressor(5)).fit(Xtr, ytr)
    err = np.linalg.norm(mdl.predict(Xte) - yte, axis=1)
    print(f'{name:8s} | z-scored kNN: median {np.median(err):5.2f} m | P90 {np.percentile(err, 90):5.2f} m')

# --- FR2 TDoA multilateration (geometry, no database): cluster arrays only, >=3 members ---
def multilaterate(dtau_row, members, tx_xy, iters=15):
    arrs = np.where(members)[0]
    ref = arrs[np.argmin(dtau_row[arrs])]                 # earliest-path reference
    others = arrs[arrs != ref]
    meas = C_LIGHT*dtau_row[others]                       # range differences [m]
    p = tx_xy[arrs].mean(0)                               # init: cluster centroid
    for _ in range(iters):
        d = np.linalg.norm(tx_xy - p, axis=1) + 1e-9
        r = (d[others] - d[ref]) - meas
        J = (p - tx_xy[others])/d[others, None] - (p - tx_xy[ref])/d[ref]
        p = p - np.linalg.lstsq(J, r, rcond=None)[0]
    return p

tx_xy = TX_POS[:, :2]
idx = np.where(ok & (mem40.sum(1) >= 3) & ~outage40)[0]
errs = []
for i in idx:
    est = multilaterate(np.nan_to_num(dtau2_q[i], posinf=0, nan=0), mem40[i], tx_xy)
    errs.append(np.linalg.norm(est - UE_POS[i, :2]))
errs = np.array(errs)
print()
print(f'FR2 TDoA multilateration: {len(idx)} UEs with >=3 cluster arrays '
      f'({len(idx)/ok.sum()*100:.0f}% of matched set)')
print(f'  error: median {np.median(errs):.2f} m | P90 {np.percentile(errs, 90):.2f} m | '
      f'sub-meter fraction {np.mean(errs < 1)*100:.1f}%')


In [ ]:
# %% [9c] Link-budget outage, NLoS diagnosis of the TDoA error, residual-gated hybrid
# (a) Cell [9b]'s outage was tautological: a threshold relative to the UE's own strongest array can
#     never fire. Detectability needs an ABSOLUTE link budget: EIRP 43 dBm, noise over 400 MHz with
#     NF 7 dB = -81 dBm  ->  an array is detectable if its channel gain exceeds ~ -124 dB.
LINK_BUDGET_DB = -124.0

heard_abs = np.isfinite(TOA2) & (E2 > LINK_BUDGET_DB)
outage_abs = ~heard_abs.any(1)
memA = (E2 >= np.where(heard_abs, E2, -np.inf).max(1, keepdims=True) + CFG.gamma_db) & heard_abs
szA = memA[ok].sum(1)
sz1 = mem1[map1[ok]].sum(1)
print(f'FR2 link-budget detectability: outage {outage_abs[ok].mean()*100:.1f}% | '
      f'mean detectable arrays {heard_abs[ok].sum(1).mean():.1f}/12')
print(f'FR2 cluster (budget-gated): median {np.median(szA):.0f} | singletons {np.mean(szA==1)*100:.1f}% '
      f'| FR1: median {np.median(sz1):.0f}, singletons {np.mean(sz1==1)*100:.1f}%')

# (b) NLoS diagnosis: excess delay = measured first-path ToA minus geometric distance / c.
#     Simulation-only oracle (uses true positions): it explains the error, it is not a feature.
d_geo = np.linalg.norm(UE_POS[:, None, :] - TX_POS[None, :, :], axis=2)      # [N, A] 3D
excess = TOA2 - d_geo/C_LIGHT                                                # [N, A] seconds
EX_THR = 3*TOA_QUANT                                                         # 7.5 ns ~ 2.25 m
err_by_class = {'all-LoS cluster': [], 'mixed/NLoS cluster': []}
res_all, err_all, is_los = [], [], []
for j, i in enumerate(idx):
    arrs = np.where(mem40[i])[0]
    worst_ex = np.nanmax(excess[i, arrs])
    est = multilaterate(np.nan_to_num(dtau2_q[i], posinf=0, nan=0), mem40[i], tx_xy)
    e = np.linalg.norm(est - UE_POS[i, :2])
    # UE-computable residual: TDoA consistency of the solution (no ground truth involved)
    ref = arrs[np.argmin(dtau2_q[i, arrs])]
    oth = arrs[arrs != ref]
    d_est = np.linalg.norm(tx_xy - est, axis=1)
    r = (d_est[oth] - d_est[ref]) - C_LIGHT*np.nan_to_num(dtau2_q[i, oth])
    res_all.append(np.sqrt(np.mean(r**2))); err_all.append(e)
    los = worst_ex < EX_THR
    is_los.append(los)
    err_by_class['all-LoS cluster' if los else 'mixed/NLoS cluster'].append(e)
res_all, err_all, is_los = np.array(res_all), np.array(err_all), np.array(is_los)
for k, v in err_by_class.items():
    v = np.array(v)
    print(f'{k:20s} {len(v):5d} UEs ({len(v)/len(idx)*100:4.1f}%) | median {np.median(v):7.2f} m '
          f'| P90 {np.percentile(v, 90):7.2f} m | sub-meter {np.mean(v < 1)*100:5.1f}%')

# (c) Residual-gated hybrid: TDoA when the UE-computable residual is small, else kNN fallback.
RES_THR = 2.0                                                                # meters
Xall = sets['FR1+FR2']; yall = y
Xtr, Xte, ytr, yte, itr, ite = train_test_split(Xall, yall, np.arange(len(yall)),
                                                test_size=0.2, random_state=CFG.seed)
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(5)).fit(Xtr, ytr)
pred_knn = knn.predict(Xte)
pos_in_idx = {int(i): j for j, i in enumerate(idx)}
hyb_err, used_tdoa = [], 0
okpos = np.where(ok)[0]
for t, row in enumerate(ite):
    i_global = int(okpos[row])
    ek = np.linalg.norm(pred_knn[t] - yte[t])
    j = pos_in_idx.get(i_global)
    if j is not None and res_all[j] < RES_THR:
        hyb_err.append(err_all[j]); used_tdoa += 1
    else:
        hyb_err.append(ek)
hyb_err = np.array(hyb_err)
kerr = np.linalg.norm(pred_knn - yte, axis=1)
print()
print(f'Hybrid (residual < {RES_THR} m -> TDoA, else kNN), test split n={len(hyb_err)}:')
print(f'  kNN alone    median {np.median(kerr):5.2f} m | P90 {np.percentile(kerr, 90):6.2f} m | sub-meter {np.mean(kerr < 1)*100:4.1f}%')
print(f'  hybrid       median {np.median(hyb_err):5.2f} m | P90 {np.percentile(hyb_err, 90):6.2f} m | '
      f'sub-meter {np.mean(hyb_err < 1)*100:4.1f}% | TDoA used on {used_tdoa/len(hyb_err)*100:.0f}% of UEs')
print(f'  residual-vs-error sanity: corr(log res, log err) over TDoA set = '
      f'{np.corrcoef(np.log10(res_all + 1e-3), np.log10(err_all + 1e-3))[0, 1]:.2f}')


In [ ]:
# %% [9d] Reference-invariant 2.5D multilateration (fixes the cell-9b solver) + hybrid, final
# Cell 9c exposed a solver bug: dtau is referenced to the ENERGY-strongest array (cell 7), while
# the geometric model was referenced to the EARLIEST one. When the two differ - routinely in ray-
# traced data, never in the original synthetic test - the mismatch enters as a systematic bias.
# Synthetic proof with a deliberately misaligned reference: old solver 18.6 m median (matching the
# observed 18.7 m on all-LoS clusters), fixed solver 0.23 m. Second fix: 2.5D geometry (known
# TX/UE heights) instead of 2D, worth 0.5-1.5 m on short links.
def multilaterate2(dtau_row, members, tx_pos, ue_z=1.5, iters=25):
    arrs = np.where(members)[0]
    ref = arrs[np.argmin(dtau_row[arrs])]
    others = arrs[arrs != ref]
    meas = C_LIGHT*(dtau_row[others] - dtau_row[ref])     # reference-INVARIANT range differences
    p = tx_pos[arrs, :2].mean(0)
    for _ in range(iters):
        q = np.array([p[0], p[1], ue_z])
        d = np.linalg.norm(tx_pos - q, axis=1) + 1e-9     # 2.5D: known heights
        r = (d[others] - d[ref]) - meas
        J = (p - tx_pos[others, :2])/d[others, None] - (p - tx_pos[ref, :2])/d[ref]
        step = np.linalg.lstsq(J, r, rcond=None)[0]
        p = p - step
        if np.linalg.norm(step) < 1e-4:
            break
    return p

CLIP_M = 300.0                                            # report divergences, don't let them overflow stats
err2, res2, los2 = [], [], []
for i in idx:
    arrs = np.where(mem40[i])[0]
    est = multilaterate2(np.nan_to_num(dtau2_q[i], posinf=0, nan=0), mem40[i], TX_POS)
    e = min(np.linalg.norm(est - UE_POS[i, :2]), CLIP_M)
    ref = arrs[np.argmin(dtau2_q[i, arrs])]
    oth = arrs[arrs != ref]
    dq = np.linalg.norm(TX_POS - np.array([est[0], est[1], 1.5]), axis=1)
    r = (dq[oth] - dq[ref]) - C_LIGHT*(np.nan_to_num(dtau2_q[i, oth]) - np.nan_to_num(dtau2_q[i, ref]))
    err2.append(e); res2.append(np.sqrt(np.mean(r**2)))
    los2.append(np.nanmax(excess[i, arrs]) < EX_THR)
err2, res2, los2 = np.array(err2), np.array(res2), np.array(los2)
clipped = int((err2 >= CLIP_M).sum())
print(f'TDoA v2 over {len(idx)} UEs ({clipped} clipped at {CLIP_M:.0f} m):')
print(f'  overall      median {np.median(err2):7.2f} m | P90 {np.percentile(err2, 90):7.2f} m | '
      f'sub-meter {np.mean(err2 < 1)*100:5.1f}%')
for lab, m in (('all-LoS', los2), ('mixed/NLoS', ~los2)):
    v = err2[m]
    print(f'  {lab:12s} {m.sum():5d} UEs | median {np.median(v):7.2f} m | '
          f'P90 {np.percentile(v, 90):7.2f} m | sub-meter {np.mean(v < 1)*100:5.1f}%')
print(f'  residual-error corr(log,log): {np.corrcoef(np.log10(res2 + 1e-3), np.log10(err2 + 1e-3))[0, 1]:.2f}')

# residual-gated hybrid with the fixed solver
pos_in_idx = {int(i): j for j, i in enumerate(idx)}
hyb2, used2 = [], 0
for t, row in enumerate(ite):
    i_global = int(okpos[row])
    j = pos_in_idx.get(i_global)
    if j is not None and res2[j] < RES_THR:
        hyb2.append(err2[j]); used2 += 1
    else:
        hyb2.append(np.linalg.norm(pred_knn[t] - yte[t]))
hyb2 = np.array(hyb2)
print()
print(f'Hybrid v2 (residual < {RES_THR} m -> TDoA, else kNN), test split n={len(hyb2)}:')
print(f'  kNN alone  median {np.median(kerr):5.2f} m | P90 {np.percentile(kerr, 90):6.2f} m | sub-meter {np.mean(kerr < 1)*100:4.1f}%')
print(f'  hybrid v2  median {np.median(hyb2):5.2f} m | P90 {np.percentile(hyb2, 90):6.2f} m | '
      f'sub-meter {np.mean(hyb2 < 1)*100:4.1f}% | TDoA used on {used2/len(hyb2)*100:.0f}% of UEs')

# diffraction status check for the confinement question (support restored in Sionna >= 1.2)
import inspect
sig = str(inspect.signature(SOLVER.__call__))
print()
print('PathSolver options:', sig)
print('NOTE: if a diffraction flag exists and defaults to False, both bands were traced WITHOUT',
      'edge diffraction and the FR1/FR2 confinement comparison is understated - re-trace to test H4a.')


In [ ]:
# %% [10] Save paired dataset + backup
drive_canary()
out = os.path.join(M3_DIR, f'paper3_{RUN_TAG}_paired.h5')
assert not os.path.exists(out), f'{out} exists — rename or delete before overwriting'
with h5py.File(out, 'w') as f:
    f.create_dataset('positions', data=UE_POS[ok])
    f.create_dataset('tx_positions', data=TX_POS.astype(np.float32))
    g2 = f.create_group('fr2')
    for k, v in dict(E_dB_q=E2_q[ok], dtau_q=dtau2_q[ok], sigma_tau=DS2[ok],
                     dphi=DPHI2[ok], sector=SECT2[ok], cluster_gamma=mem2[ok],
                     outage=outage2[ok]).items():
        g2.create_dataset(k, data=v)
    g1 = f.create_group('fr1')
    for k, v in dict(E_dB_q=E1_q[i1], dtau_q=dtau1_q[i1], sigma_tau=ds1[i1],
                     cluster_gamma=mem1[i1]).items():
        g1.create_dataset(k, data=v)
    f.attrs.update(dict(fr2_carrier_hz=CFG.carrier_hz, fr2_bandwidth_hz=CFG.bandwidth_hz,
                        gamma_db=CFG.gamma_db, seed=CFG.seed, n_sectors=CFG.n_sectors))
print('Saved:', out, f'({os.path.getsize(out)/1e6:.1f} MB)')

import shutil
zip_path = shutil.make_archive('/content/paper3_m3_backup', 'zip', M3_DIR)
from google.colab import files as colab_files
colab_files.download(zip_path)